____
### 0. Preamble
- This serves as a continuation to my other notebook file which trained models in a monopolistic environment. 
- Within this notebook file, I will be finetuning the hyperparameters of the 3 models and pitting themselves in the same environment to simulate oligopolistic environments

____
### 1. Imports

In [4]:
import gymnasium as gym
import numpy as np
import torch as torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import copy

____
### 2. Importing Original Environment

In [18]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.spaces.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32)
        #action is a discrete number between 5 to 50
        self.latest = np.array([self.max_inventory, self.max_steps, 0], dtype=np.float32)

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        self.latest = self.obs()
        return self.obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action[0]) #obtain the price as a float
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #sell out 

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        reward = reward / 100.0
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps
        if truncated and self.inventory > 0:
            reward -= self.inventory * 2.0  # $2 penalty per unsold unit
        self.latest = self.obs()
        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory / self.max_inventory, # 0 to 1
                        (self.max_steps - self.step_count) / self.max_steps, # 0 to 1
                        self.last_demand / self.max_inventory] # 0 to 1
                        , dtype=np.float32)

    def get_latest(self):
        obs = self.latest
        print(f"Leftover Stock: {obs[0]} units, Days Left {obs[1]}, Sold Units: {obs[2]}")

    def demand(self, price):
        base = 40
        sensitivity = 0.8
        noise = np.random.normal(0, 2) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

____
### 3. Importing classes, models
- This is a generic Agent class that contains shared methods that are used across all classes

In [5]:
class Agent:
    def __init__(self, env=None, name="bot"):
        self.name = name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.env = env if env is not None else DynamicPricingEnv()
        self.obs_dim = self.env.observation_space.shape[0]
        self.action_dim = self.env.action_space.shape[0]
        self.action_low = float(self.env.action_space.low[0])
        self.action_high = float(self.env.action_space.high[0])

    def rescale(self, raw_action):
        return self.action_low + (raw_action + 1.0) * 0.5 * (self.action_high - self.action_low)

    def action_to_price(self, action):
        return float(action[0]) if hasattr(action, '__iter__') else float(action)

    def play(self, render=False, deterministic=True):
        obs, _ = self.env.reset()
        done = False
        total_reward = 0.0
        total_revenue = 0.0
        steps = 0
        trajectory = []
        while not done:
            action = self.act(obs, deterministic=deterministic)
            price = self.action_to_price(action)
            next_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            demand = int(getattr(self.env, 'last_demand', 0))
            sold = demand
            revenue = price * sold
            trajectory.append((obs, price, demand, float(reward)))
            print(f"Step {steps+1}: Demand: {demand}, Price: {price:.2f}, Reward: {reward:.2f}, Sold: {sold}, Revenue: {revenue:.2f}")
            total_reward += reward
            total_revenue += revenue
            steps += 1
            obs = next_obs
            if render:
                self.env.get_latest()
        print(f"Episode finished - Reward: {total_reward:.2f}, Steps: {steps}, Ending inventory: {int(self.env.inventory)}")
        print(f"Total revenue generated: ${total_revenue:.2f}")
        return {"total_reward": total_reward, "total_revenue": total_revenue, "steps": steps,
                "ending_inventory": int(self.env.inventory), "trajectory": trajectory}

    def evaluate(self, n_episodes=100):
        rewards, inventories, steps = [], [], []
        for _ in range(n_episodes):
            env = DynamicPricingEnv()
            obs, _ = env.reset()
            done = False
            total_reward = 0.0
            while not done:
                action = self.act(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                done = terminated or truncated
            rewards.append(total_reward)
            inventories.append(env.inventory)
            steps.append(env.step_count)
        print(f"Episodes: {n_episodes}")
        print(f"Mean reward: {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}")
        print(f"Reward range: [{np.min(rewards):.2f}, {np.max(rewards):.2f}]")
        print(f"Mean ending inventory: {np.mean(inventories):.2f}")
        print(f"Mean steps: {np.mean(steps):.2f}")

    def save(self, path=None):
        if path is None:
            path = f"{self.name}_{self.__class__.__name__}.pth"
        payload = {
            "class_name": self.__class__.__name__,
            "name": self.name,
            "modules": {attr: module.state_dict() for attr, module in self.__dict__.items() if isinstance(module, nn.Module)},
            "optimizer_states": {attr: obj.state_dict() for attr, obj in self.__dict__.items() if hasattr(obj, "state_dict") and attr.endswith("_opt")},
        }
        torch.save(payload, path)
        return path

    def load(self, path, map_location=None):
        payload = torch.load(path, map_location=map_location if map_location is not None else self.device)
        modules = payload.get("modules", {})
        for attr, state_dict in modules.items():
            module = getattr(self, attr, None)
            if isinstance(module, nn.Module):
                module.load_state_dict(state_dict)
        optimizer_states = payload.get("optimizer_states", {})
        for attr, state_dict in optimizer_states.items():
            optimizer = getattr(self, attr, None)
            if hasattr(optimizer, "load_state_dict"):
                optimizer.load_state_dict(state_dict)
        return self

#### 3.1 PPO Agent


In [19]:
class PPOActorCritic(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__() #initialises the parent nn.Module class
        # self.backbone acts as a shared extractor used by both the actor and critic network
        self.backbone = nn.Sequential( #nn.Sequential runs the layers in order, linear -> Tanh -> linear -> tanh
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )
        self.actor_mean = nn.Linear(64, action_dim) 
        self.log_std = nn.Parameter(torch.ones(action_dim) * 2.0) 
        self.critic = nn.Linear(64, 1)

    def forward(self, obs): #needs to be overridden
        features = self.backbone(obs) #vector of 128 values
        mean = self.actor_mean(features) #obtain the action 
        std = self.log_std.exp().expand_as(mean) #obtain the std dev and fits the shape with with mean 
        critic_val = self.critic(features) #obtain the critic value
        return mean, std, critic_val
    
    def get_action(self, obs): #creating action based off the network
        mean, std, value = self.forward(obs) #calling forward to obtain values from actor and critic network
        dist = Normal(mean, std) #creates normal distribution
        action = dist.sample() #samples the distribution
        log_prob = dist.log_prob(action).sum(dim=-1) #obtains sum of log distribution (exp below)
        return action, log_prob, value.squeeze(-1) #converts the value into a scalar

    def evaluate(self, obs, action):
        mean, std, value = self.forward(obs)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1) #calculates the entropy of the normal distribution, how random or uncertain the distribution is 
        # entropy is summed up over the last dimension
        return log_prob, value.squeeze(-1), entropy

In [20]:
class RolloutBuffer:
    def __init__(self):
        self.clear() #delegates initialisation to the clear() method
    def clear(self): #resets all lists to empty
        self.obs, self.actions, self.log_probs = [], [], []
        self.rewards, self.values, self.dones  = [], [], []
    def add(self, obs, action, log_prob, reward, value, done): #appends one round of experiences
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)
    def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95): #compute generalized advantage est
        advantages = [] 
        gae = 0.0
        values = self.values + [last_value] # adds one extra value to compute next-step difference
        for t in reversed(range(len(self.rewards))):
            delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
            gae   = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, self.values)]
        return advantages, returns
    def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns

In [ ]:
class PPOAgent(Agent):
    def __init__(self, name: str = "bot", NN=None, env: gym.Env = None): #Store a PyTorch model and device for inference / persistence
        super().__init__(env=env, name=name)
        if NN is None:
            self.NN = PPOActorCritic(self.obs_dim, self.action_dim).to(self.device)
        else:
            self.NN = NN.to(self.device)
        self.buffer = RolloutBuffer()
        self.optimizer = optim.Adam(self.NN.parameters(), lr=3e-4)

    def train(self, total_timesteps=200_000, n_steps=512, render=True):
        self.NN.train() #set to training mode
        obs, _ = self.env.reset()
        episode_reward, episode_count = 0 , 0
        timestep = 0
        while timestep < total_timesteps: # runs until timestep budget is exhausted
            self.buffer.clear() #reset the buffer at the start of every rollout
            for _ in range(n_steps):
                obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    action, log_prob, value = self.NN.get_action(obs_tensor)
                action_np = action.cpu().numpy()[0]
                action_np = np.clip(action_np, self.action_low, self.action_high)
                next_obs, reward, terminated, truncated, _ = self.env.step(action_np)
                done = terminated or truncated
                self.buffer.add(obs = obs,
                        action = action.squeeze(0).cpu(),
                        log_prob = log_prob.squeeze(0).cpu(),
                        reward = reward,
                        value = value.squeeze(0).cpu().item(),
                        done = float(done))
                episode_reward += reward
                obs = next_obs
                timestep += 1
                if done:
                    episode_count += 1
                    if episode_count % 20 == 0 and render == True:
                        print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                            f"Reward: {episode_reward:>8.2f}")
                    episode_reward = 0
                    obs, _  = self.env.reset()
            with torch.no_grad():
                last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)
                _, _, last_value = self.NN.get_action(last_obs)
                last_value = last_value.squeeze(0).cpu().item()
            advantages, returns = self.buffer.compute_returns(last_value)
            obs_t, act_t, lp_t, adv_t, ret_t = self.buffer.to_tensors(advantages, returns, self.device)
            self.ppo_update(obs_t, act_t, lp_t, adv_t, ret_t)
        print("Training complete.")
        return self

    def act(self, obs, deterministic=True):
        obs_tensor = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.NN.eval()
        with torch.no_grad():
            mean, std, _ = self.NN.forward(obs_tensor)
            if deterministic:
                action = mean
            else:
                action = Normal(mean, std).sample()
        action_np = action.squeeze(0).cpu().numpy()
        action_np = np.clip(action_np, self.action_low, self.action_high).astype(np.float32)
        return action_np

    #helpers:
    def ppo_update(self, obs, actions, old_log_probs, advantages, returns, 
               clip_range=0.2, ent_coef=0.05, vf_coef=0.5, n_epochs=10, batch_size=64):
        total_steps = obs.shape[0]
        self.NN.train()
        for _ in range(n_epochs):
            indices = torch.randperm(total_steps)
            for start in range(0, total_steps, batch_size):
                idx = indices[start : start + batch_size]
                new_log_probs, values, entropy = self.NN.evaluate(obs[idx], actions[idx])
                ratio = (new_log_probs - old_log_probs[idx]).exp()
                adv = advantages[idx]
                policy_loss = -torch.min(ratio * adv, torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv).mean()
                value_loss = nn.functional.mse_loss(values, returns[idx])
                entropy_loss = -entropy.mean()
                loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.NN.parameters(), max_norm=0.5)
                self.optimizer.step()

NameError: name 'gym' is not defined

#### 3.2 TD3 Agent

In [22]:
class TD3Actor(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential( 
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh()
        )
        
    def forward(self, obs): #needs to be overridden
        return self.net(obs)

In [23]:
class TD3Critic(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim + action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, obs, action):
        x = torch.cat([obs, action], dim=-1)
        return self.net(x)

In [24]:
class ReplayBuffer:
    def __init__(self, capacity=100000): 
        self.capacity = capacity
        self.buffer = []
        self.position = 0 #tracks where to write the next transition

    def add(self, obs, action, reward, next_obs, done):
        transition = (obs, action, reward, next_obs,done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.position] = transition
            self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        obs, actions, rewards, next_obs, dones = zip(*batch)
        return (torch.tensor(np.array(obs), dtype=torch.float32),
                torch.tensor(np.array(actions), dtype=torch.float32),
                torch.tensor(np.array(rewards), dtype=torch.float32).unsqueeze(1),
                torch.tensor(np.array(next_obs), dtype=torch.float32),
                torch.tensor(np.array(dones), dtype=torch.float32).unsqueeze(1))
    def __len__(self):
        return len(self.buffer)

In [ ]:
class TD3Agent(Agent):
    def __init__(self, env=None, gamma=0.99, tau=0.005, policy_noise=0.2, noise_clip=0.5,
                 policy_delay=2, expl_noise=0.1, batch_size=256, buffer_capacity=100_000, lr=1e-3):
        super().__init__(env=env)
        obs_dim = self.obs_dim   # 3
        action_dim = self.action_dim
        #initialising the 3 main networks
        self.actor = TD3Actor(obs_dim, action_dim).to(self.device)
        self.critic1 = TD3Critic(obs_dim, action_dim).to(self.device)
        self.critic2 = TD3Critic(obs_dim, action_dim).to(self.device)
        # initialising target networks, frozen copies which are updated slowly via polyak
        self.tgt_actor   = copy.deepcopy(self.actor)
        self.tgt_critic1 = copy.deepcopy(self.critic1)
        self.tgt_critic2 = copy.deepcopy(self.critic2)
        # target networks never receive gradient updates directly
        for net in [self.tgt_actor, self.tgt_critic1, self.tgt_critic2]:
            for param in net.parameters():
                param.requires_grad = False
        self.actor_opt   = optim.Adam(self.actor.parameters(), lr=lr) #optimisers
        self.critic1_opt = optim.Adam(self.critic1.parameters(), lr=lr)
        self.critic2_opt = optim.Adam(self.critic2.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(capacity=buffer_capacity) #buffer
        # hyperparameters
        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_delay = policy_delay
        self.expl_noise = expl_noise
        self.batch_size = batch_size
        self.total_updates = 0  # tracks how many critic updates done, for policy_delay

    def act(self, obs, deterministic=False): #method to produce an action given an observation 
        obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.actor.eval()
        with torch.no_grad():
            raw = self.actor(obs_t)# tanh output in (-1, 1)
        self.actor.train()
        price = self.rescale(raw.cpu().numpy()[0])
        if not deterministic:
            noise = np.random.normal(0, self.expl_noise *
                                     (self.action_high - self.action_low),
                                     size=price.shape)
            price = np.clip(price + noise, self.action_low, self.action_high)
        return price.astype(np.float32)

    def td3_update(self): #function to update the model during training 
        obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(self.batch_size) #unpacks a random batch selected from the replay buffer
        obs = obs.to(self.device) # move everything to the correct device
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_obs = next_obs.to(self.device)
        dones = dones.to(self.device)
        with torch.no_grad(): #to update critic
            raw_next = self.tgt_actor(next_obs) # target actor suggests the next action from next_obs within range (-1, 1)
            next_actions = self.rescale(raw_next)
            noise = torch.randn_like(next_actions) * self.policy_noise
            noise = noise.clamp(-self.noise_clip, self.noise_clip)
            next_actions = (next_actions + noise).clamp(self.action_low, self.action_high)
            q1_next = self.tgt_critic1(next_obs, next_actions) # twin critics evaluate (next_obs, next_actions)
            q2_next = self.tgt_critic2(next_obs, next_actions)
            q_next = torch.min(q1_next, q2_next) #use min as the target
            target_q = rewards + self.gamma * (1.0 - dones) * q_next # bellman target: if episode ended (done=1), no future reward
        q1_current = self.critic1(obs, actions) # compute current Q estimates and MSE against the target
        q2_current = self.critic2(obs, actions)
        critic1_loss = nn.functional.mse_loss(q1_current, target_q)
        critic2_loss = nn.functional.mse_loss(q2_current, target_q)
        self.critic1_opt.zero_grad() # backpropagate critic 1
        critic1_loss.backward()
        self.critic1_opt.step()
        self.critic2_opt.zero_grad() # backpropagate critic 2
        critic2_loss.backward()
        self.critic2_opt.step()
        self.total_updates += 1
        if self.total_updates % self.policy_delay == 0: #update the policy
            raw_actions = self.actor(obs)
            actor_actions = self.rescale(raw_actions)
            actor_loss = -self.critic1(obs, actor_actions).mean() # gradient ascent on Q → gradient descent on negative Q
            self.critic1_opt.zero_grad()
            self.actor_opt.zero_grad()
            actor_loss.backward()
            self.actor_opt.step()  # only actor weights are updated, not critic1
            for main, target in [(self.actor,   self.tgt_actor), (self.critic1, self.tgt_critic1), (self.critic2, self.tgt_critic2)]:#polyak updates, slowly updating the TGT network
                for p_main, p_tgt in zip(main.parameters(), target.parameters()):
                    p_tgt.data.mul_(1.0 - self.tau)
                    p_tgt.data.add_(self.tau * p_main.data)
    
    def train(self, total_timesteps=200_000, learning_starts=1_000, log_every=20, render=True): #function to train the model 
        obs, _ = self.env.reset()
        episode_reward = 0.0
        episode_count = 0
        for timestep in range(1, total_timesteps + 1):
            # before learning_starts, take random actions to pre-fill buffer
            if timestep < learning_starts:
                action = self.env.action_space.sample()
            else:
                action = self.act(obs, deterministic=False)
            next_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            # store transition — use terminated (not done) for the done flag
            # so that a truncated episode doesn't incorrectly zero out future rewards
            self.replay_buffer.add(obs, action, reward, next_obs, float(terminated))
            episode_reward += reward
            obs = next_obs
            if done:
                episode_count += 1
                if episode_count % log_every == 0 and render == True:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f}")
                episode_reward = 0.0
                obs, _ = self.env.reset()
            # only start learning once the buffer has enough samples
            if timestep >= learning_starts:
                self.td3_update()
        print("Training complete.")
        return self

#### 3.3 SAC Agent

In [26]:
class SACActor(nn.Module):
    def __init__(self, obs_dim, action_dim, log_std_min=-20, log_std_max=2):
        super().__init__()
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max 
        self.net = nn.Sequential( #shared backbone network 
            nn.Linear(obs_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU())
        self.mean_NN = nn.Linear(256, action_dim) #network to output the head
        self.log_std_NN = nn.Linear(256, action_dim) #network to output the log_std

    def forward(self, obs):
        features = self.net(obs)
        mean = self.mean_NN(features)
        log_std = self.log_std_NN(features)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mean, log_std

    def sample(self, obs):
        mean, log_std = self.forward(obs)
        std = log_std.exp()
        normal = Normal(mean, std)
        x = normal.rsample()
        action = torch.tanh(x)
        log_prob = normal.log_prob(x) - torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)  # sum over action dimensions
        return action, log_prob

In [27]:
class SACCritic(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1))

    def forward(self, obs, action):
        x = torch.cat([obs, action], dim=-1)
        return self.net(x)

In [6]:
class SACAgent(Agent):
    def __init__(self, env=None, gamma=0.99, tau=0.005, lr=3e-4, batch_size=256, 
                 buffer_capacity=100_000, target_entropy=None, alpha_lr=3e-4):
        super().__init__(env=env)
        obs_dim = self.obs_dim # 3
        action_dim = self.action_dim # 1
        #initialising the 3 main networks
        self.actor = SACActor(obs_dim, action_dim).to(self.device)
        self.critic1 = SACCritic(obs_dim, action_dim).to(self.device)
        self.critic2 = SACCritic(obs_dim, action_dim).to(self.device)
        # initialising target network, frozen copies which are updated slowly via polyak the stochastic actor provides its own smoothing through sampling
        self.tgt_critic1 = copy.deepcopy(self.critic1)
        self.tgt_critic2 = copy.deepcopy(self.critic2)
        #target networks do not receive gradient updates directly
        for net in [self.tgt_critic1, self.tgt_critic2]:
            for param in net.parameters():
                param.requires_grad = False
        self.actor_opt   = optim.Adam(self.actor.parameters(), lr=lr) #optimisers
        self.critic1_opt = optim.Adam(self.critic1.parameters(), lr=lr)
        self.critic2_opt = optim.Adam(self.critic2.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(capacity=buffer_capacity) #buffer
        if target_entropy is not None: #target minimum level of randomness the actor should maintain throughout training
            self.target_entropy = target_entropy
        else:
            self.target_entropy = -float(action_dim)
        self.log_alpha = torch.zeros(1, requires_grad=True, device=self.device) #learnable temperature parameter stored in log space
        self.alpha_opt = optim.Adam([self.log_alpha], lr=alpha_lr) #optimiser for learnable temperature parameter
        # hyperparameters
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size

    def act(self, obs, deterministic=False): # method to obtain action based off observation
        obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.actor.eval()
        with torch.no_grad():
            if deterministic:
                mean, _ = self.actor.forward(obs_t)
                raw = torch.tanh(mean)
            else:
                raw, _ = self.actor.sample(obs_t)
        self.actor.train()
        price = self.rescale(raw.cpu().numpy()[0])
        price = np.clip(price, self.action_low, self.action_high)
        return price.astype(np.float32)
    
    def sac_update(self):
        obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(self.batch_size)
        obs = obs.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_obs = next_obs.to(self.device)
        dones = dones.to(self.device)
        alpha = self.log_alpha.exp().detach() 
        with torch.no_grad():
            next_raw, next_log_prob = self.actor.sample(next_obs)
            next_actions = self.rescale(next_raw)
            q1_next = self.tgt_critic1(next_obs, torch.tensor(next_actions, dtype=torch.float32, device=self.device))
            q2_next = self.tgt_critic2(next_obs, torch.tensor(next_actions, dtype=torch.float32, device=self.device))
            q_next = torch.min(q1_next, q2_next)
            target_q = rewards + self.gamma * (1.0 - dones) * (q_next - alpha * next_log_prob)
        q1_current = self.critic1(obs, actions)
        q2_current = self.critic2(obs, actions)
        critic1_loss = nn.functional.mse_loss(q1_current, target_q)
        critic2_loss = nn.functional.mse_loss(q2_current, target_q)
        self.critic1_opt.zero_grad()
        critic1_loss.backward()
        self.critic1_opt.step()
        self.critic2_opt.zero_grad()
        critic2_loss.backward()
        self.critic2_opt.step()
        raw, log_prob = self.actor.sample(obs)
        actor_actions = self.rescale(raw)
        actor_actions_t = torch.tensor(actor_actions, dtype=torch.float32, device=self.device)
        q1_actor = self.critic1(obs, actor_actions_t)
        q2_actor = self.critic2(obs, actor_actions_t)
        min_q    = torch.min(q1_actor, q2_actor)
        actor_loss = (alpha * log_prob - min_q).mean()
        self.critic1_opt.zero_grad()  # clear stale critic gradients
        self.critic2_opt.zero_grad()
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()
        alpha_loss = -(self.log_alpha * (log_prob.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()
        for main, target in [(self.critic1, self.tgt_critic1),
                              (self.critic2, self.tgt_critic2)]:
            for p_main, p_tgt in zip(main.parameters(), target.parameters()):
                p_tgt.data.mul_(1.0 - self.tau)
                p_tgt.data.add_(self.tau * p_main.data)

    def train(self, total_timesteps=200_000, learning_starts=1_000, log_every=20, render = True): #method to train the model
        obs, _ = self.env.reset()
        episode_reward, episode_count = 0.0, 0
        for timestep in range(1, total_timesteps + 1):
            if timestep < learning_starts:
                action = self.env.action_space.sample()
            else:
                action = self.act(obs, deterministic=False)
            next_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            self.replay_buffer.add(obs, action, reward, next_obs, float(terminated))
            episode_reward += reward
            obs = next_obs
            if done:
                episode_count += 1
                if episode_count % log_every == 0 and render == True:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f} | Alpha: {self.log_alpha.exp().item():.4f}")
                episode_reward = 0.0
                obs, _ = self.env.reset()
            if timestep >= learning_starts:
                self.sac_update()
        print("Training complete.")
        return self

### 4. Training each model

In [29]:
"""
ppo = PPOAgent()
td3 = TD3Agent()
sac = SACAgent()
model_lst = [ppo, td3, sac]

for model in model_lst:
    model.train()
    model.evaluate()
"""

'\nppo = PPOAgent()\ntd3 = TD3Agent()\nsac = SACAgent()\nmodel_lst = [ppo, td3, sac]\n\nfor model in model_lst:\n    model.train()\n    model.evaluate()\n'

### 5. Creating a new environment that can fit all 3 models 

In [30]:
class OligopolyEnv(gym.Env):
    def __init__(self, agents, agent_idx): #agents: list of the 3 trained/training agent objects agent_idx: which agent "owns" this env instance
        self.agents = agents # reference to all agents stored globally
        self.agent_idx = agent_idx # this env's owner
        self.n_agents = len(agents) #number of agents in this environment
        self.max_steps = 30 
        self.max_inventory = 100
        self.cost = 5.0
        self.total_market = 40 # total market demand base
        self.sensitivity = 0.8 #how strongly demand drops when avg price increases
        self.observation_space = gym.spaces.Box(low=np.array([0, 0, 0]), #numbers from 0 to 1
                                                high=np.array([1, 1, 1]), 
                                                dtype=np.float32) # represents the inventory, remaining time and last demand
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32) #price charged, discrete number from 5-50 
        self.inventories = None #to store inventory of all agents
        self.step_count = 0 # tracks time in episode
        self.last_demands = None #stores last demand for each agent 

    def reset(self, seed=None, options=None):
        self.inventories = [self.max_inventory] * self.n_agents # each agent starts with 100 inventories , [100, 100, 100]
        self.step_count = 0 
        self.last_demands = [0] * self.n_agents #each agent starts with 0 last known demand, [0, 0, 0]
        return self.obs(), {}

    def step(self, action): #Called by agent `agent_idx`. Collects actions from all other agents first, then resolves the market simultaneously.
        my_price = float(np.clip(action[0], 5.0, 50.0)) #extracts the price and clips to range (5, 50)
        # Get prices from all other agents (deterministic inference)
        prices = [None] * self.n_agents #initialises empty list
        prices[self.agent_idx] = my_price #stores own price
        for i, agent in enumerate(self.agents): #loop through all agents
            if i != self.agent_idx: # skip owner of instance
                other_obs = self.agents[i].env.obs() #get other agent's observation
                prices[i] = float(np.clip(agent.act(other_obs, deterministic=False)[0], 5.0, 50.0)) #stores value of other agent's prices
        shares = self.compute_shares(prices) #market share via softmin (lower price → larger share)
        total_demand = self.total_demand(prices) # Resolve demand and update inventories
        rewards = []
        for i in range(self.n_agents):
            demand_i = int(total_demand * shares[i] + np.random.normal(0, 1))
            demand_i = max(0, demand_i)
            units_sold = min(demand_i, self.inventories[i])
            self.inventories[i] -= units_sold
            self.last_demands[i] = units_sold
            profit = (prices[i] - self.cost) * units_sold / 100.0
            rewards.append(profit)
        self.step_count += 1
        terminated = self.inventories[self.agent_idx] <= 0
        truncated = self.step_count >= self.max_steps
        if truncated and self.inventories[self.agent_idx] > 0:
            rewards[self.agent_idx] -= self.inventories[self.agent_idx] * 2.0
        return self.obs(), rewards[self.agent_idx], terminated, truncated, {}

    def obs(self):
        i = self.agent_idx
        return np.array([ #normalising the values so that it ranges from 0 to 1 for all 3 values of observation space
            self.inventories[i] / self.max_inventory,
            (self.max_steps - self.step_count) / self.max_steps,
            self.last_demands[i] / self.max_inventory
        ], dtype=np.float32)

    def total_demand(self, prices): # Market-level demand based on average price
        avg_price = np.mean(prices)
        return max(0, self.total_market - self.sensitivity * avg_price
                   + np.random.normal(0, 2))

    def compute_shares(self, prices): # Softmin: lower price wins more share
        exps = np.array([np.exp(-0.1 * p) for p in prices])
        return exps / exps.sum()

In [ ]:
ppo = PPOAgent()
td3 = TD3Agent()
sac = SACAgent()
agents = [ppo, td3, sac]

def train(agents):
    for i, agent in enumerate(agents):
        agent.env = OligopolyEnv(agents, agent_idx = i)
        agent.env.reset()

    for round in range(5):  
        for agent in agents:
            agent.env.reset()
            agent.train(total_timesteps=50_000, render = False)
            agent.save()


Timestep     129 | Episode   20 | Reward:     0.00
Timestep     256 | Episode   40 | Reward:     1.32
Timestep     385 | Episode   60 | Reward:     0.63
Timestep     516 | Episode   80 | Reward:     0.00
Timestep     647 | Episode  100 | Reward:     3.29
Timestep     779 | Episode  120 | Reward:     1.50
Timestep     915 | Episode  140 | Reward:     0.00
Timestep    1049 | Episode  160 | Reward:     2.69
Timestep    1184 | Episode  180 | Reward:     2.63
Timestep    1326 | Episode  200 | Reward:     0.41
Timestep    1459 | Episode  220 | Reward:     1.22
Timestep    1602 | Episode  240 | Reward:     1.90
Timestep    1741 | Episode  260 | Reward:     5.72
Timestep    1878 | Episode  280 | Reward:     0.45
Timestep    2021 | Episode  300 | Reward:     4.03
Timestep    2164 | Episode  320 | Reward:     3.82
Timestep    2317 | Episode  340 | Reward:     4.85
Timestep    2459 | Episode  360 | Reward:     3.27
Timestep    2603 | Episode  380 | Reward:     3.52
Timestep    2754 | Episode  400

C:\Users\xuan2\AppData\Local\Temp\ipykernel_24772\3677660048.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  q1_next = self.tgt_critic1(next_obs, torch.tensor(next_actions, dtype=torch.float32, device=self.device))
C:\Users\xuan2\AppData\Local\Temp\ipykernel_24772\3677660048.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  q2_next = self.tgt_critic2(next_obs, torch.tensor(next_actions, dtype=torch.float32, device=self.device))
C:\Users\xuan2\AppData\Local\Temp\ipykernel_24772\3677660048.py:80: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceT

Timestep    1172 | Episode   80 | Reward:    10.22 | Alpha: 0.9497
Timestep    1477 | Episode  100 | Reward:    10.55 | Alpha: 0.8666
Timestep    1755 | Episode  120 | Reward:    14.72 | Alpha: 0.7973
Timestep    2027 | Episode  140 | Reward:    10.35 | Alpha: 0.7348
Timestep    2303 | Episode  160 | Reward:    14.09 | Alpha: 0.6764
Timestep    2594 | Episode  180 | Reward:    13.14 | Alpha: 0.6198
Timestep    2879 | Episode  200 | Reward:    12.67 | Alpha: 0.5691
Timestep    3160 | Episode  220 | Reward:    11.24 | Alpha: 0.5231
Timestep    3453 | Episode  240 | Reward:    14.25 | Alpha: 0.4790
Timestep    3743 | Episode  260 | Reward:    10.14 | Alpha: 0.4391
Timestep    4030 | Episode  280 | Reward:     9.84 | Alpha: 0.4029
Timestep    4302 | Episode  300 | Reward:     9.62 | Alpha: 0.3713
Timestep    4603 | Episode  320 | Reward:    18.76 | Alpha: 0.3392
Timestep    4879 | Episode  340 | Reward:    11.48 | Alpha: 0.3123
Timestep    5167 | Episode  360 | Reward:    13.14 | Alpha: 0.